# Notebook 06 — Validade da resposta ao item Q23b

**Item:** "Eu me sinto motivado(a), no dia a dia, a usar o que foi ensinado." (questão 23b do questionário do aluno; escala de 0 = concordo totalmente a 3 = discordo totalmente). Estudantes que **discordam** do item têm proficiência mais alta.

Este notebook examina se esse padrão é específico do item ou se aparece em toda a bateria de perguntas.

- **Estilo de resposta:** estudantes com menor proficiência em leitura tenderiam a concordar mais com qualquer afirmação (aquiescência) ou a usar mais os extremos da escala.
- **Perfil crítico:** estudantes com maior proficiência avaliariam de forma mais crítica a escola e a própria motivação.

**Expectativas definidas antes da análise.** Os nove itens da questão 23 são afirmações no mesmo sentido: concordar é sempre a resposta favorável. Dois itens permitem um contraste: **23e** ("Eu me sinto à vontade para discordar dos meus professores") e **23f** ("Eu consigo argumentar sobre conteúdos difíceis") descrevem uma postura mais crítica à escola.

- Se predominar o estilo de resposta, estudantes de maior proficiência tenderão a discordar mais de todos os itens, inclusive 23e e 23f.
- Se predominar o perfil crítico, eles tenderão a discordar mais dos itens de avaliação da escola e da motivação, mas a concordar mais com 23e e 23f.

| Seção | Conteúdo |
| --- | --- |
| 1 | Os itens das questões 22 e 23 |
| 2 | Relação de cada item com a proficiência |
| 3 | Índices de estilo de resposta |
| 4 | O coeficiente da Q23b controlado pelo estilo de resposta |
| 5 | As categorias "discordo" e "discordo totalmente" |

**Entrada:** `base_analitica.parquet`. **Saída:** `tabelas_q23b.xlsx`.

## 0. Configuração

In [1]:
import sys, numpy as np, pandas as pd
import statsmodels, statsmodels.formula.api as smf
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/dissertacao/'   # ajuste para a sua pasta
except ImportError:
    PASTA = './'

FATOR_SAEB = 55.093
print({'python': sys.version.split()[0], 'pandas': pd.__version__, 'statsmodels': statsmodels.__version__})
base = pd.read_parquet(PASTA + 'base_analitica.parquet').reset_index(drop=True)
assert len(base) == 22996
# Q23b sem preenchimento: respostas ausentes voltam a ser NaN nesta análise
base['q23_aluno_motivacao_aplicar_aprendizado'] = base['aluno_motivacao_aplicar_aprendizado'].where(
    base['ausente_aluno_motivacao_aplicar_aprendizado'] == 0)
TABELAS = {}

def fmt_p(p):
    return '< 0,001' if p < 0.001 else f'{p:.3f}'.replace('.', ',')
def ajustar(f, dados):
    return smf.ols(f, dados).fit(cov_type='cluster', cov_kwds={'groups': dados['id_escola']}, use_t=True)

Mounted at /content/drive
{'python': '3.13.16', 'pandas': '2.2.3', 'statsmodels': '0.15.0'}


## 1. Os itens

In [2]:
Q23 = {'q23_aluno_interesse_conteudos': ('23a', 'Eu me interesso sobre o que foi ensinado na escola neste ano.', 'Motivação'),
       'q23_aluno_motivacao_aplicar_aprendizado': ('23b', 'Eu me sinto motivado(a), no dia a dia, a usar o que foi ensinado.', 'Motivação'),
       'q23_sala_espaco_diferentes_opinioes': ('23c', 'Há espaço para diferentes opiniões na minha sala de aula.', 'Avaliação da escola'),
       'q23_aluno_sente_seguranca_escola': ('23d', 'Eu me sinto seguro(a) quando estou na escola.', 'Avaliação da escola'),
       'q23_aluno_conforto_discordar_professor': ('23e', 'Eu me sinto à vontade para discordar dos(as) meus(minhas) professores(as).', 'Postura crítica'),
       'q23_aluno_capacidade_argumentacao': ('23f', 'Eu consigo argumentar sobre conteúdos difíceis.', 'Postura crítica'),
       'q23_avaliacao_representa_aprendizado': ('23g', 'Os resultados das avaliações representam o quanto eu aprendi.', 'Avaliação da escola'),
       'q23_professor_acredita_capacidade_aluno': ('23h', 'Meus(minhas) professores(as) acreditam que eu sou capaz de aprender.', 'Avaliação da escola'),
       'q23_professor_motiva_continuidade_estudos': ('23i', 'Meus(minhas) professores(as) me motivam a continuar meus estudos.', 'Avaliação da escola')}
Q22 = {'q22_professor_informa_conteudo_inicio_ano': ('22a', 'No início do ano, informaram o que seria ensinado'),
       'q22_professor_investiga_conhecimento_previo': ('22b', 'Perguntam o que os estudantes já sabem antes de um novo conteúdo'),
       'q22_professor_contextualiza_conteudo_cotidiano': ('22c', 'Trazem temas do cotidiano para debate'),
       'q22_professor_aborda_desigualdade_racial': ('22d', 'Abordam desigualdade racial'),
       'q22_professor_aborda_desigualdade_genero': ('22e', 'Abordam desigualdade de gênero'),
       'q22_professor_aborda_bullying_violencia': ('22f', 'Abordam bullying e outras formas de violência'),
       'q22_professor_desenvolve_trabalho_grupo': ('22g', 'Desenvolvem trabalhos em grupo'),
       'q22_professor_aborda_futuro_profissional': ('22h', 'Abordam o futuro profissional dos estudantes')}
print('Questão 23: escala 0 = concordo totalmente, 1 = concordo, 2 = discordo, 3 = discordo totalmente.')
print('Questão 22: escala 0 = todos os professores, 1 = a maior parte, 2 = poucos, 3 = nenhum.')
print('Em ambas, valores maiores correspondem a respostas menos favoráveis.')

Questão 23: escala 0 = concordo totalmente, 1 = concordo, 2 = discordo, 3 = discordo totalmente.
Questão 22: escala 0 = todos os professores, 1 = a maior parte, 2 = poucos, 3 = nenhum.
Em ambas, valores maiores correspondem a respostas menos favoráveis.


## 2. Relação de cada item com a proficiência

Para cada item o percentual de concordância (ou de "todos/a maior parte" na questão 22), proficiência média por categoria e inclinação — a diferença média de proficiência, em pontos SAEB, a cada passo da escala em direção à resposta menos favorável (regressão simples com erros-padrão agrupados por escola). Inclinação positiva significa que respostas menos favoráveis estão associadas a proficiência mais alta.

In [3]:
def linha(col, cod, texto, grupo):
    d = base[[col, 'proficiencia', 'proficiencia_saeb', 'id_escola']].dropna()
    r = ajustar(f'proficiencia ~ {col}', d)
    ic = r.conf_int().loc[col] * FATOR_SAEB
    medias = d.groupby(col)['proficiencia_saeb'].mean()
    return {'Item': cod, 'Afirmação': texto, 'Grupo': grupo, 'N válido': len(d),
            'Favorável (%)': round(100 * (d[col] <= 1).mean(), 1),
            **{f'Média na categoria {int(k)}': round(v, 1) for k, v in medias.items()},
            'Inclinação (pontos por categoria)': round(r.params[col] * FATOR_SAEB, 1),
            'IC 95%': f'{ic[0]:.1f} a {ic[1]:.1f}'.replace('.', ','), 'p-valor': fmt_p(r.pvalues[col])}

t23 = pd.DataFrame([linha(c, *v) for c, v in Q23.items()])
t22 = pd.DataFrame([linha(c, cod, txt, 'Práticas dos professores') for c, (cod, txt) in Q22.items()])
TABELAS['Itens Q23'] = t23; TABELAS['Itens Q22'] = t22
cols = ['Item', 'Grupo', 'Favorável (%)', 'Inclinação (pontos por categoria)', 'IC 95%', 'p-valor']
print('Questão 23:'); print(t23[cols].to_string(index=False))
print('\nQuestão 22:'); print(t22[cols].to_string(index=False))

Questão 23:
Item               Grupo  Favorável (%)  Inclinação (pontos por categoria)      IC 95% p-valor
 23a           Motivação           86.1                                6.6   5,4 a 7,8 < 0,001
 23b           Motivação           61.2                               12.5 11,3 a 13,6 < 0,001
 23c Avaliação da escola           77.5                                3.6   2,6 a 4,6 < 0,001
 23d Avaliação da escola           70.5                               -0.7  -1,9 a 0,4   0,217
 23e     Postura crítica           56.5                                2.8   1,8 a 3,7 < 0,001
 23f     Postura crítica           61.2                               -1.1 -2,0 a -0,2   0,021
 23g Avaliação da escola           60.2                               12.3 11,5 a 13,1 < 0,001
 23h Avaliação da escola           90.9                               -3.3 -4,4 a -2,2 < 0,001
 23i Avaliação da escola           87.7                                0.7  -0,4 a 1,8   0,186

Questão 22:
Item                    G

## 3. Índices de estilo de resposta

Calculados com os oito itens da questão 23 **exceto** a 23b, para cada estudante com pelo menos cinco respostas válidas:

- **Concordância geral (aquiescência):** proporção de itens respondidos com "concordo" ou "concordo totalmente".
- **Uso dos extremos:** proporção de itens respondidos com "concordo totalmente" ou "discordo totalmente".
- **Respostas iguais em todos os itens:** estudante que marcou a mesma alternativa em todos os itens respondidos.

In [4]:
outros = [c for c in Q23 if c != 'q23_aluno_motivacao_aplicar_aprendizado']
M = base[outros]
validos = M.notna().sum(axis=1)
ok = validos >= 5
base['concordancia_geral'] = ((M <= 1).sum(axis=1) / validos).where(ok)
base['uso_extremos'] = (M.isin([0, 3]).sum(axis=1) / validos).where(ok)
base['media_outros_itens'] = M.mean(axis=1).where(ok)
base['respostas_iguais'] = (M.nunique(axis=1) == 1).astype(float).where(ok)

print(f'Estudantes com pelo menos 5 respostas válidas: {ok.sum()} de {len(base)}')
ind = []
for col, rot in [('concordancia_geral', 'Concordância geral (0 a 1)'), ('uso_extremos', 'Uso dos extremos (0 a 1)'),
                 ('respostas_iguais', 'Respostas iguais em todos os itens (0/1)')]:
    d = base[[col, 'proficiencia', 'id_escola']].dropna()
    r = ajustar(f'proficiencia ~ {col}', d)
    ind.append({'Índice': rot, 'Média': round(d[col].mean(), 3),
                'Correlação de Spearman com a proficiência': round(spearmanr(d[col], d['proficiencia']).statistic, 3),
                'Diferença de proficiência entre 0 e 1 (pontos SAEB)': round(r.params[col] * FATOR_SAEB, 1),
                'p-valor': fmt_p(r.pvalues[col])})
ind = pd.DataFrame(ind)
TABELAS['Índices de estilo'] = ind
print(ind.to_string(index=False))

Estudantes com pelo menos 5 respostas válidas: 22849 de 22996
                                  Índice  Média  Correlação de Spearman com a proficiência  Diferença de proficiência entre 0 e 1 (pontos SAEB) p-valor
              Concordância geral (0 a 1)  0.739                                     -0.077                                                -11.5 < 0,001
                Uso dos extremos (0 a 1)  0.272                                     -0.038                                                -14.3 < 0,001
Respostas iguais em todos os itens (0/1)  0.109                                     -0.140                                                -22.5 < 0,001


## 4. O coeficiente da Q23b controlado pelo estilo de resposta

O modelo principal (Notebook 03) é reestimado **nos mesmos estudantes** em quatro versões. Se o coeficiente da Q23b diminuir muito ao incluir os índices de estilo, parte da associação se deve à forma de responder, e não ao conteúdo do item. A última versão substitui a Q23b pelo seu **componente específico**: a resposta à 23b menos a média do estudante nos outros itens.

In [5]:
ATRIB = ("C(sexo, Treatment('Masculino')) + C(raca_cor, Treatment('Branca')) + "
         "C(pretensao_futura, Treatment('Somente estudar')) + idade + reprovacao + tempo_estudos + tempo_cursos + "
         "tempo_trabalho_domestico + tempo_trabalho + mora_mae + mora_pai + escolaridade_mae + escolaridade_pai + "
         "responsaveis_incentivam_estudo + responsaveis_conversam_escola + qtd_computadores + possui_wifi + "
         "possui_escrivaninha + possui_casa_quarto_individual + em_integrado + nao_sabe_escolaridade_mae + "
         "nao_sabe_escolaridade_pai + ausente_escolaridade_mae + ausente_escolaridade_pai + ausente_mora_mae + "
         "ausente_mora_pai + n_itens_ausentes")
Q = 'q23_aluno_motivacao_aplicar_aprendizado'
d = base.dropna(subset=[Q, 'concordancia_geral', 'uso_extremos', 'media_outros_itens']).copy()
d['q23b_especifico'] = d[Q] - d['media_outros_itens']
versoes = {'1. Modelo principal (mesmos estudantes)': f'proficiencia ~ {Q} + ' + ATRIB,
           '2. + concordância geral e uso dos extremos': f'proficiencia ~ {Q} + concordancia_geral + uso_extremos + ' + ATRIB,
           '3. + média nos outros itens da questão 23': f'proficiencia ~ {Q} + media_outros_itens + ' + ATRIB}
linhas = []
for nome, f in versoes.items():
    r = ajustar(f, d); ic = r.conf_int().loc[Q] * FATOR_SAEB
    linhas.append({'Versão': nome, 'Coeficiente da Q23b (pontos SAEB por categoria)': round(r.params[Q] * FATOR_SAEB, 1),
                   'IC 95%': f'{ic[0]:.1f} a {ic[1]:.1f}'.replace('.', ','), 'p-valor': fmt_p(r.pvalues[Q]),
                   'R²': round(r.rsquared, 4)})
r = ajustar('proficiencia ~ q23b_especifico + media_outros_itens + ' + ATRIB, d)
ic = r.conf_int().loc['q23b_especifico'] * FATOR_SAEB
linhas.append({'Versão': '4. Componente específico da Q23b + média nos outros itens',
               'Coeficiente da Q23b (pontos SAEB por categoria)': round(r.params['q23b_especifico'] * FATOR_SAEB, 1),
               'IC 95%': f'{ic[0]:.1f} a {ic[1]:.1f}'.replace('.', ','), 'p-valor': fmt_p(r.pvalues['q23b_especifico']),
               'R²': round(r.rsquared, 4)})
r2 = ajustar(versoes['2. + concordância geral e uso dos extremos'], d)
modelos = pd.DataFrame(linhas)
TABELAS['Q23b controlada'] = modelos
print(f'N = {len(d)}')
print(modelos.to_string(index=False))
print(f"\nNo modelo 2: concordância geral = {r2.params['concordancia_geral'] * FATOR_SAEB:.1f} pontos (p {fmt_p(r2.pvalues['concordancia_geral'])}); "
      f"uso dos extremos = {r2.params['uso_extremos'] * FATOR_SAEB:.1f} pontos (p {fmt_p(r2.pvalues['uso_extremos'])})")

N = 22753
                                                   Versão  Coeficiente da Q23b (pontos SAEB por categoria)      IC 95% p-valor     R²
                  1. Modelo principal (mesmos estudantes)                                             11.2 10,3 a 12,2 < 0,001 0.2060
               2. + concordância geral e uso dos extremos                                             12.2 11,1 a 13,2 < 0,001 0.2077
                3. + média nos outros itens da questão 23                                             12.7 11,5 a 13,8 < 0,001 0.2071
4. Componente específico da Q23b + média nos outros itens                                             12.7 11,5 a 13,8 < 0,001 0.2071

No modelo 2: concordância geral = 9.4 pontos (p < 0,001); uso dos extremos = -3.6 pontos (p 0,003)


## 5. As categorias "discordo" e "discordo totalmente"

Diferença de proficiência entre as duas categorias de discordância da Q23b, com intervalo de confiança (erros-padrão agrupados por escola).

In [6]:
dd = base[base[Q].isin([2, 3])].assign(disc_total=lambda x: (x[Q] == 3).astype(int))
r = ajustar('proficiencia ~ disc_total', dd); ic = r.conf_int().loc['disc_total'] * FATOR_SAEB
cat = base.groupby(Q)['proficiencia_saeb'].agg(['size', 'mean']).round(1)
cat.index = ['0 = concordo totalmente', '1 = concordo', '2 = discordo', '3 = discordo totalmente']
print(cat.rename(columns={'size': 'N', 'mean': 'Média (pontos SAEB)'}).to_string())
print(f"\nDiscordo totalmente − discordo: {r.params['disc_total'] * FATOR_SAEB:.1f} pontos "
      f"(IC 95%: {ic[0]:.1f} a {ic[1]:.1f}; p {fmt_p(r.pvalues['disc_total'])})")
TABELAS['Categorias Q23b'] = cat.reset_index().rename(columns={'index': 'Categoria', 'size': 'N', 'mean': 'Média (pontos SAEB)'})

                             N  Média (pontos SAEB)
0 = concordo totalmente   2609                247.5
1 = concordo             11338                266.8
2 = discordo              7344                280.4
3 = discordo totalmente   1504                280.1

Discordo totalmente − discordo: -0.3 pontos (IC 95%: -3.5 a 2.9; p 0,860)


## Exportação

In [7]:
with pd.ExcelWriter(PASTA + 'tabelas_q23b.xlsx') as xw:
    for nome, t in TABELAS.items():
        t.to_excel(xw, sheet_name=nome[:31], index=False)
print('Tabelas salvas em', PASTA + 'tabelas_q23b.xlsx')

Tabelas salvas em /content/drive/MyDrive/dissertacao/tabelas_q23b.xlsx
